# Tokenization với PhoBERT

## Goal

Đọc một câu đã được word segmentation từ PostgreSQL và kiểm tra cách PhoBERT tạo
`input_ids`, `attention_mask` và token embeddings. Token chỉ là dữ liệu trung gian nên
notebook không lưu token IDs hoặc mask vào database.

## Setup

Chạy notebook từ thư mục `backend/`.

In [1]:
import torch
from transformers import AutoModel, AutoTokenizer

from information_retrieval.infrastructure.config import get_settings
from information_retrieval.infrastructure.database import create_database_engine
from information_retrieval.infrastructure.sentence_embedding_repository import (
    PostgresSentenceEmbeddingRepository,
)

/Users/thangtran/Workplace/master_s_degree/information_retrieval/.worktrees/sentence-embedding-pipeline/backend/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
settings = get_settings()
engine = create_database_engine(settings.database_url)
repository = PostgresSentenceEmbeddingRepository(engine)

MODEL_NAME = settings.phobert_model_name
CACHE_DIR = settings.phobert_model_dir
MAX_LENGTH = settings.embedding_max_length

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
phobert = AutoModel.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
phobert.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 57746.40it/s]


[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(64001, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(258, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=Tru

## Steps

### 1. Lấy một câu thật từ corpus

In [3]:
sentences = repository.list_for_embedding()
if not sentences:
    raise RuntimeError("Database chưa có segmented_sentences; hãy chạy make segment trước.")

sample = sentences[0]
sentence = sample.segmented_text

print(f"crawl_url_id={sample.crawl_url_id} segmented_sentence_id={sample.id}")
print(sentence)

crawl_url_id=1 segmented_sentence_id=22
Đại_biểu muốn gọi xe cấp_cứu thuận_tiện như gọi taxi


### 2. Tokenization

In [4]:
encoded = tokenizer(
    sentence,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH,
)

input_ids = encoded["input_ids"][0]
attention_mask = encoded["attention_mask"][0]
tokens = tokenizer.convert_ids_to_tokens(input_ids)

print(f"{'TOKEN':<25} {'ID':<10} {'MASK':<5}")
print("-" * 45)
for token, token_id, mask_value in zip(
    tokens, input_ids.tolist(), attention_mask.tolist(), strict=True
):
    print(f"{token:<25} {token_id:<10} {mask_value:<5}")

TOKEN                     ID         MASK 
---------------------------------------------
<s>                       0          1    
Đại_biểu                  4420       1    
muốn                      202        1    
gọi                       328        1    
xe                        105        1    
cấp_cứu                   1629       1    
thuận_tiện                4798       1    
như                       42         1    
gọi                       328        1    
taxi                      2367       1    
</s>                      2          1    


### 3. Forward và mean pooling

In [5]:
with torch.no_grad():
    output = phobert(**encoded)

token_embeddings = output.last_hidden_state
special_tokens_mask = torch.tensor(
    [
        tokenizer.get_special_tokens_mask(ids, already_has_special_tokens=True)
        for ids in encoded["input_ids"].tolist()
    ]
)
content_mask = encoded["attention_mask"] * (1 - special_tokens_mask)
pooling_mask = content_mask.unsqueeze(-1).float()
sentence_embedding = (token_embeddings * pooling_mask).sum(dim=1) / pooling_mask.sum(dim=1).clamp(
    min=1e-9
)

print("Token embeddings:", token_embeddings.shape)
print("Sentence embedding:", sentence_embedding.shape)
print("Truncated at max length:", len(input_ids) == MAX_LENGTH)
print("Finite embedding:", torch.isfinite(sentence_embedding).all().item())

Token embeddings: torch.Size([1, 11, 768])
Sentence embedding: torch.Size([1, 768])
Truncated at max length: False
Finite embedding: True


## Checks

- `input_ids` và `attention_mask` có cùng chiều dài.
- Token embeddings có shape `[1, số_token, 768]`.
- Sentence embedding có shape `[1, 768]` và chỉ pooling content tokens.

## Next Steps

Notebook `03_document_embedding_v1.ipynb` áp dụng cùng quy tắc tokenization theo batch và
lưu một vector cho mỗi câu vào pgvector.